# Ridewise - Feature Engineering
- Transformed clean rider and Trip dataset can help have a better churn prediction
- engineer 20+ features
- RFM framwork + behavioural + temporal signals

Step 1 - setup

In [24]:
# Import Libraries
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# configurations ngs("ignore")
%matplotlib inline
warnings.filterwarnings
sns.set_theme(style="whitegrid", font_scale=1.2)

# define the location of our raw and processed data
DATA_PROCESSED = os.path.join("..", "data", "processed")

# Load the Datasets
riders = pd.read_csv(os.path.join(DATA_PROCESSED,"riders_clean.csv"),parse_dates=["signup_date"])
drivers = pd.read_csv(os.path.join(DATA_PROCESSED,"drivers_clean.csv"),parse_dates=["signup_date"])
trips = pd.read_csv(os.path.join(DATA_PROCESSED,"trips_clean.csv"),parse_dates=["pickup_time", "dropoff_time"])
sessions = pd.read_csv(os.path.join(DATA_PROCESSED,"sessions_clean.csv"),parse_dates=["session_time"])

# Reference data
# In real ML deployment settings, reference_date is mean Now
# we calculate " days since last trip"

reference_date = trips["pickup_time"].max()

### step 1a - Engineering some rider features
- 1- Eligible Riders (AKA active): check for riders who had atleast one trips
- 2- check for riders with trip in the last 60 days

Note: this can be used for promotions or campaigns


In [25]:
# Eligible Riders
riders_with_trips = set(trips["user_id"])

# check account age in days
riders["account_age_days"] =(
    reference_date - riders["signup_date"]
).dt.days

#Eligible riders
eligible = riders[
    (riders["user_id"].isin(riders_with_trips)) &
    (riders["account_age_days"] >= 60)
][["user_id", "churned"]].copy()

Step 1 b - Engineering RFM features
- RFM-Recency, Frequency, Mnetary(Revenue metrics)

In [26]:
 # Recency: how long since last trip

# group 1: Last trip per user
last_trip = (
    trips.groupby("user_id")["pickup_time"].max()
    .reset_index(name="last_trip")
)

# group 2: last session date per user
last_session = (
    sessions.groupby("rider_id")["session_time"].max()
    .reset_index()
    .rename(columns={"session_time": "last_session", "rider_id": "user_id"})
)

# recency : combination of group 1 and group 2
recency = (
    eligible[["user_id"]]
    .merge(last_trip, on="user_id", how="left")
    .merge(last_session, on="user_id", how="left")
)

# calculate the recency features
recency["days_since_last_trip"] = (reference_date - recency["last_trip"]).dt.days
recency["days_since_last_session"] = (reference_date - recency["last_session"]).dt.days

# Final recency features
Recency = recency[["user_id", "days_since_last_trip", "days_since_last_session"]].copy()

recency.head()

,user_id,last_trip,last_session,days_since_last_trip,days_since_last_session
0,R00000,2025-04-02 14:46:29,2025-04-27 16:06:46,25,0.0
1,R00001,2025-04-22 04:35:17,2025-04-27 10:33:25,5,0.0
2,R00002,2025-04-13 00:08:00,2025-04-27 15:04:35,14,0.0
3,R00004,2025-04-15 05:30:04,2025-04-27 18:28:35,12,0.0
4,R00005,2025-04-25 15:12:08,2025-04-27 18:09:20,2,0.0


In [27]:
# step 1- Frequency: How many trips over a rolling window (e.g., last 7, 30, 60, 90 days)
frequency = eligible[["user_id"]].copy()

# step 2 define the rolling windows
for days in [7, 30, 60, 90]:
    cutoff = reference_date - pd.Timedelta(days=days)
    
    counts = (
        trips[trips["pickup_time"] >= cutoff]
        .groupby("user_id", as_index=False)
        .size()  # counts trips per user
        .rename(columns={"size": f"trips_last_{days}"})
    )
    
    # step 3- merge counts into frequency
    frequency = frequency.merge(counts, on="user_id", how="left")
    
    # step 4- clean up the "NO show"
    frequency[f"trips_last_{days}"] = frequency[f"trips_last_{days}"].fillna(0)

    frequency.head()

In [33]:
# Monetary: revenue matrics: lifetime revenue

# 3 monetary feature
# 1-(monetary total)---lifetime revenue: high value customers= higher retention priority(perks and discount)
# 2- (monetary average)-- avg fare per trip: budget or premium user
# 3-(monetary_last_30d): recent spend -- high trips + low spend= budget ride
# low_trips + high spend 

# Set reference date
reference_date = trips["pickup_time"].max()

# Lifetime monetary metrics
monetary = (
    trips.groupby("user_id")
    .agg(
        monetary_total=("total_revenue", "sum"),
        monetary_avg=("total_revenue", "mean"),
        trips_lifetime=("trip_id", "count")
    )
    .round(2)
    .reset_index()
)

# Monetary in the last 30 days
cutoff_30 = reference_date - pd.Timedelta(days=30)

monetary_30 = (
    trips[trips["pickup_time"] >= cutoff_30]
    .groupby("user_id")["total_revenue"]
    .sum()
    .reset_index(name="monetary_last_30d")
)

# Merge
monetary = monetary.merge(monetary_30, on="user_id", how="left")
monetary["monetary_last_30d"] = monetary["monetary_last_30d"].fillna(0)

monetary.head()

KeyError: "Column(s) ['total_revenue'] do not exist"

Step 2- RFM Score: Quantile Binning

- Group these riders based on how often they take trips(Grouping)

- problem : its hard to group then because they have continuous values

- put riders into buckets or categories

In [29]:
# step 2a - RFM score: RFM- tells us how valuable a rider is

# step 2b - scoring logic: what are the criteria in putting a rider in a bin?

# step 2c - solution with RFM score (put the rider data(metric) into a scale e.g 1-5)

# step 2d - Quantile Binning

# mean: divide the riders int 5 different groups (20% each) and assign a score to them

# step 2e - Recency score --- Lower days = Better ( 5day vs 40days) 

# step 2f - Frequency Score

# step 2g - Monetary Score

# step 2h - final RFM score (RS+FS+MS)

# step 2i - why does this matter to churn prediction
# churn rate by RFM combine score

# score | interpretation | churn rate
# 5.0       excellent rider   2% rarely churn
# 4.0       loyal rider       5% churn rate
# 1.0       lost rider        60% churn rate
# 3.0        At risk          15% churn rate


In [32]:
# RFM Scores(1-5 quantile bins)

# Merge R + F + M tables
rfm = (
    recency[["user_id", "days_since_last_trip"]]
    .merge(frequency[["user_id", "trips_last_30"]], on="user_id", how="left")
    .merge(monetary[["user_id", "monetary_total"]], on="user_id", how="left")
    .fillna({"trips_last_30": 0, "monetary_total": 0})
)

# Recency Score
rfm["rfm_recency_score"] = pd.qcut(
    rfm["days_since_last_trip"].rank(method="first"),
    q=5,
    labels=[5,4,3,2,1]
).astype(int)

# Frequency Score
rfm["rfm_frequency_score"] = pd.qcut(
    rfm["trips_last_30"].rank(method="first"),
    q=5,
    labels=[1,2,3,4,5]
).astype(int)

# Monetary Score
rfm["rfm_monetary_score"] = pd.qcut(
    rfm["monetary_total"].rank(method="first"),
    q=5,
    labels=[1,2,3,4,5]
).astype(int)

# Combined RFM score
rfm["rfm_combined_score"] = (
    rfm["rfm_recency_score"]
    + rfm["rfm_frequency_score"]
    + rfm["rfm_monetary_score"]
) / 3

rfm.head()

NameError: name 'monetary' is not defined

In [31]:
print(recency.columns)
print(frequency.columns)
print(monetary.columns)

Index(['user_id', 'last_trip', 'last_session', 'days_since_last_trip',
       'days_since_last_session'],
      dtype='object')
Index(['user_id', 'trips_last_7', 'trips_last_30', 'trips_last_60',
       'trips_last_90'],
      dtype='object')


NameError: name 'monetary' is not defined